# Providence, RI — Land Value Tax Model

**City real-property levy, revenue-neutral 4:1 split-rate applied within each rate class, all current classes and exemptions preserved.**

Rhode Island has no county government and Providence funds its own schools, so the city levy is the entire property tax on
Providence real estate. Real property is assessed at **100% of market value** (revaluation as of December 31, 2024). The FY2026
levy ordinance (Code of Ordinances § 21-182) taxes real estate at **seven class rates** — owner-occupied vs. non-owner-occupied
residential, larger apartment buildings, and commercial/industrial — and taxes mixed-use Class 3 property at a blend (residential
rate on the residential portion, commercial rate on the rest).

This model keeps that structure intact. The 4:1 split (land taxed at 4× the improvement rate) is solved **separately and
revenue-neutrally inside each rate class** (the Rochester dual-rate pattern), so owner-occupied homes keep their lower rates,
commercial property keeps its higher rate, and no revenue moves between classes. Only the land/building split changes.

| | |
|---|---|
| Geographic scope | City of Providence (Providence County, FIPS 44007) |
| Levy modeled | City real-property tax, FY2026 bills (assessment date Dec 31, 2024). Tangible personal property (Class 4, $53.40) is not modeled |
| Reform | Revenue-neutral 4:1 split-rate within each of the 7 rate classes |
| Assessment | 100% of market value; no assessment ratio |
| Rates (per $1,000) | Owner-occ. 1 unit 8.40 · owner-occ. 2–5 units 7.55 · non-owner-occ. 1 unit 14.60 · non-owner-occ. 2–5 units 14.00 · 6–10 units 26.00 · 11+ units 28.50 · commercial/industrial 29.20 (FY2026 Approved Ordinance Book, § 21-182(c)) |
| Current tax | Actual FY2026 bills (`total_taxes`) from the city's 2025 Property Tax Roll |
| Validation | FY2026 roll billed total excluding TSA contract taxes; the same computation on the FY2025 roll reproduces the audited FY2025 ACFR real-property levy |
| Data sources | City GIS `PVD_Radius_Parcels_FL` (parcels + CAMA land/building values); Providence Open Data `6ub4-iebe` (2025 Property Tax Roll) and `xvti-7dtw` (2024 roll, back-test only) |

## Section 1 — Imports and Setup

In [ ]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'providence'
STATE_FIPS = '44'                 # Rhode Island
COUNTY_FIPS = '007'               # Providence County
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

# FY2026 real-property rates per $1,000 of assessed value, keyed by the roll's levy_code_1
# (FY2026 Approved Ordinance Book, Code of Ordinances § 21-182(c))
RATE_CLASSES = {
    'OO01': 8.40,     # Class 1A owner-occupied (1 unit, incl. owner-occupied condos)
    'OO2-5': 7.55,    # Class 1B owner-occupied 2-5 units
    'NO01': 14.60,    # Class 1A non-owner-occupied (incl. non-owner-occupied condos, residential vacant land)
    'NOO2-5': 14.00,  # Class 1B non-owner-occupied 2-5 units
    'C610': 26.00,    # Class 1C 6-10 units
    'C11': 28.50,     # Class 1D 11+ units
    'C01': 29.20,     # Class 2A commercial/industrial, plus Class 3 mixed-use billed at a blend
}
# Levy codes whose tax is set by contract or statute rather than by assessed value: held out of the reform
OUT_OF_SCOPE_LEVIES = {
    'TSA': 'Tax stabilization agreement (RIGL 44-3-9), contractual tax',
    '8LAW': 'Low-income housing taxed at 8% of gross rent (RIGL 44-5-13.11)',
    'PRA': 'Non-standard levy code (3 accounts)',
    'R01': 'Non-standard levy code (1 account)',
}
# Audited FY2025 real-property levy (ACFR, Exhibit C-1 p. 107, reported in $ thousands) — Gate 1 back-test control
ACFR_FY2025_REAL_PROPERTY_LEVY = 315_771_000

# Parcel-map export (lvt/parcel_map.py) — per-parcel identity columns carried onto the
# GeoParquet + interactive map but NOT the flat cross-city CSV.
PARCEL_ID_COL = 'PROPID'          # plat-lot-unit, e.g. 043-0586-0000 (= roll tax_map)
OWNER_NAME_COL = 'Owner1'
OWNER_ADDRESS_COL = 'ParcAddress'  # situs address
# The assessor's Vision portal (gis.vgsi.com/providenceri) keys parcels by an internal pid. No deep link
# could be verified against PROPID, the roll p_id or the GIS AV_PID, so this stays None rather than emit
# links that may 404.
PARCEL_URL_TEMPLATE = None

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load Parcel Data

Two sources of the same vintage (assessment date Dec 31, 2024 → FY2026 bills), plus the prior-year roll for the back-test:

| Source | Use |
|---|---|
| City GIS FeatureServer `PVD_Radius_Parcels_FL/0` (44,365 polygons, `TaxRollYear` 2025) | Geometry, CAMA `AssessedValueLand` / `AssessedValueBuildings`, use code, units, owner, situs address |
| Providence Open Data `6ub4-iebe` — 2025 Property Tax Roll (44,372 rows) | Rate class (`levy_code_1`), roll class, assessed value, exemptions, **billed tax** (`total_taxes`) |
| Providence Open Data `xvti-7dtw` — 2024 Property Tax Roll | FY2025 bills, used only for the Gate 1 back-test against the audited FY2025 ACFR |

The Socrata description of `6ub4-iebe` says "billed FY2025", but its bills reproduce the FY2026 ordinance rates (Section 4), so it
is the FY2026 roll. The GIS layer is loaded through the FeatureServer (requesting EPSG:4326), not the equivalent "Parcels with
CAMA" MapServer, because `get_mapserver_data_with_geometry` hardcodes EPSG:3435. Do **not** use the
`services2.arcgis.com/qvkbeam7Wirps6zC/.../parcel_file_current` layer that web search returns for "Providence parcels": it is Detroit.

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels_cama.gpq'
ROLL_PATH = DATA_DIR / 'tax_roll_2025.csv'
ROLL_PRIOR_PATH = DATA_DIR / 'tax_roll_2024.csv'
GIS_BASE_URL = 'https://webgis.providenceri.gov/server/rest/services'
ROLL_URL = 'https://data.providenceri.gov/resource/6ub4-iebe.csv?$limit=100000'
ROLL_PRIOR_URL = 'https://data.providenceri.gov/resource/xvti-7dtw.csv?$limit=100000'

if PARCEL_PATH.exists():
    parcels = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(parcels):,} parcel features from cache")
else:
    parcels = get_feature_data_with_geometry(
        dataset_name='PVD_Radius_Parcels_FL',
        base_url=GIS_BASE_URL,
        layer_id=0,
        paginate=True,
        verbose=False,
    )
    parcels.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(parcels):,} parcel features")


def load_roll(path: Path, url: str) -> pd.DataFrame:
    # Socrata property tax roll, cached locally; all columns read as strings
    if not path.exists():
        pd.read_csv(url, dtype=str).to_csv(path, index=False)
    return pd.read_csv(path, dtype=str)


roll_raw = load_roll(ROLL_PATH, ROLL_URL)
roll_prior_raw = load_roll(ROLL_PRIOR_PATH, ROLL_PRIOR_URL)
print(f"2025 roll: {len(roll_raw):,} rows   2024 roll: {len(roll_prior_raw):,} rows")

In [ ]:
MONEY_COLS = ['total_assmt', 'total_exempt', 'total_taxes']


def dedupe_roll(raw: pd.DataFrame) -> pd.DataFrame:
    # The export repeats accounts (same p_id, differing only in situs/point columns): keep one row per account
    roll = raw.drop_duplicates('p_id').copy()
    for col in MONEY_COLS:
        roll[col] = pd.to_numeric(roll[col], errors='coerce').fillna(0.0)
    return roll


roll = dedupe_roll(roll_raw)
roll_prior = dedupe_roll(roll_prior_raw)
print(f"2025 roll: dropped {len(roll_raw) - len(roll):,} duplicate rows -> {len(roll):,} accounts, "
      f"${roll['total_taxes'].sum():,.0f} billed")

# GIS: numeric values, drop blank-PROPID ROW/water slivers ($0), dissolve multi-part parcels
for col in ['AssessedValueLand', 'AssessedValueBuildings', 'AssessedValueTotal', 'NumUnits', 'AreaSF']:
    parcels[col] = pd.to_numeric(parcels[col], errors='coerce')
parcels['PROPID'] = parcels['PROPID'].fillna('').str.strip()
n_blank = int((parcels['PROPID'] == '').sum())
blank_value = parcels.loc[parcels['PROPID'] == '', 'AssessedValueTotal'].fillna(0).sum()
parcels = parcels[parcels['PROPID'] != ''].copy()
n_features = len(parcels)
# Multi-part parcels arrive as several features sharing one PROPID with identical values
parcels = parcels.dissolve(by='PROPID', aggfunc='first').reset_index()
print(f"GIS: dropped {n_blank:,} blank-PROPID slivers (${blank_value:,.0f} value); "
      f"{n_features:,} features -> {len(parcels):,} parcels")

## Section 3 — Join, Classify, and Fix the Condo Land Values

**Join.** Roll `tax_map` (e.g. `043-0586-0000`) equals GIS `PROPID`, and assessed totals agree on every joined account. The GIS
layer has kept updating since the assessment date (plat changes into 2026), so a few hundred roll accounts whose lots were later
merged or re-platted have no polygon. Their replacement lots are in GIS at $0. The land/building split exists only in GIS, so
those accounts cannot be modeled; they are dropped and counted below.

**Scope.** The reform applies to the seven ordinance rate classes. Held out, and excluded from the export and charts:
- `E01` fully-exempt parcels, plus rate-class accounts whose personal exemptions cover their whole value;
- `TSA` tax stabilization agreements and `8LAW` low-income housing taxed on rent. Their taxes are fixed by contract or statute,
  not by assessed value, so a land/building rate split does not reach them;
- `PRA` / `R01`, four accounts on non-standard levy codes.

**Condo land (Gate 5 canary, fixed here).** Every condo unit, roll class 23 (residential) and 24 (commercial), carries
`AssessedValueLand = 0`: the assessor books all condo value as building, and no account on a condo lot carries the lot's land.
Left alone, condos would be taxed as pure buildings and show a large, false tax cut. A few other taxable building accounts share
the problem because they sit on a condo lot: the $67M Blue Cross building shares plat-lot 019-0102 with 199 condo units, and a
$52M hotel shares 019-0143 with 61 condos and an apartment account holding under 1% of the lot's value as land. By contrast, most
other $0-land building accounts (suffixes like `LB01`, `BLDG`) are **buildings on leased land** whose lot is valued on a separate
ground account, taxable or exempt. Those are correctly assessed. Under a split-rate they pay only the building rate while the
ground account carries the land tax, so they are left as assessed.

The fix therefore works per **plat-lot** (`PROPID[:8]`; condo units of one complex share one lot polygon), on lots holding a
taxable building account with $0 land:

1. **Density-aware target:** lot area (union of the lot's polygons, EPSG:3438, ft²) × the median land $/ft² of the 20 nearest
   taxable non-condo parcels, as a share of the total assessed value of every account on the lot. Towers on small lots get a
   small land share.
2. **Ceiling:** no higher than the median land share of the 20 nearest improved taxable non-condo parcels. A condo complex is
   at least as intensively built as its neighbors.
3. **Floor:** no lower than the 10th-percentile land share of Providence's improved 6+ unit apartment and mixed-use buildings.
   This guards against lots whose neighbors are unusually cheap land (the Residences at Providence Place on the mall parcel, the
   Rising Sun Mills conversion).
4. **Only under-landed lots, only the missing land:** a lot is imputed only when the land booked on all its accounts is below that
   floor share of the lot's value. The missing land (target − booked) goes to the lot's $0-land taxable accounts pro rata by
   assessed value.

Each re-split account gets `land = total × share` and `building = total − land`. Its total assessed value, exemptions, and
current tax are unchanged; only the land/building split moves. Units stay as rows because units of one complex sit in different
rate classes (owner- vs. non-owner-occupied). The simpler neighborhood-median-share approach (St. Paul) was rejected because it
ignores density: it gives condos a ~49% value-weighted land share, far above Providence's own multi-family buildings (2–5 family
~28%, 11+ units ~17%).

**Categories** come from the roll's class, which is authoritative for the tax year (mapping in code below). Providence's Class 1B
is 2–5 units, so 5- and 6-unit buildings in it (per GIS `NumUnits`) go to Large Multi-Family. The GIS carries no finer commercial
use code (`StateUse` is blank), so retail, office and hotel are not separated and stay in `Commercial`. "CI Vacant Land" includes
commercial lots with minor site improvements such as paving, and stays `Vacant Land`.

In [ ]:
gdf = parcels.merge(
    roll[['p_id', 'tax_map', 'class', 'short_desc', 'levy_code_1'] + MONEY_COLS],
    left_on='PROPID', right_on='tax_map', how='inner',
)
unjoined = roll[~roll['tax_map'].isin(parcels['PROPID'])]
value_match = np.isclose(gdf['total_assmt'], gdf['AssessedValueTotal']).mean()
print(f"Joined {len(gdf):,} of {len(roll):,} roll accounts; assessed totals match on {value_match:.1%}")
print(f"Unjoined roll accounts (lot re-platted since assessment): {len(unjoined):,}, "
      f"${unjoined['total_taxes'].sum():,.0f} billed")
print(unjoined.groupby('levy_code_1')['total_taxes'].agg(['size', 'sum']).round(0).to_string())

gdf['net_assessed'] = gdf['total_assmt'] - gdf['total_exempt']
in_rate_class = gdf['levy_code_1'].isin(RATE_CLASSES.keys())
# TSA/8LAW accounts also carry exemptions equal to their value (the device that applies the contract tax),
# so "exemption covers the whole value" only means fully exempt inside the rate classes
gdf['full_exmp'] = ((gdf['levy_code_1'] == 'E01') | (in_rate_class & (gdf['net_assessed'] <= 0))).astype(int)
gdf['in_reform'] = in_rate_class & (gdf['full_exmp'] == 0)

scope_group = np.select(
    [gdf['in_reform'], gdf['full_exmp'] == 1, gdf['levy_code_1'].isin(OUT_OF_SCOPE_LEVIES.keys())],
    ['Reform: 7 rate classes', 'Held out: fully exempt', 'Held out: ' + gdf['levy_code_1']],
    default='Other',
)
scope = gdf.groupby(scope_group).agg(
    accounts=('PROPID', 'size'), assessed=('total_assmt', 'sum'), billed=('total_taxes', 'sum'))
print()
print(scope.to_string(float_format=lambda v: f"{v:,.0f}"))

In [ ]:
CONDO_CLASSES = ['23', '24']
KNN_K = 20


def knn_median(qx, qy, rx, ry, values, k=KNN_K, chunk=200):
    # Median of `values` over the k nearest reference points to each query point (planar distance)
    out = np.empty(len(qx))
    for start in range(0, len(qx), chunk):
        d2 = (qx[start:start + chunk, None] - rx[None, :]) ** 2 + (qy[start:start + chunk, None] - ry[None, :]) ** 2
        nearest = np.argpartition(d2, k, axis=1)[:, :k]
        out[start:start + chunk] = np.median(values[nearest], axis=1)
    return out


proj = gdf.to_crs(3438)  # NAD83 / Rhode Island (ftUS)
gdf['lot_sf'] = proj.geometry.area.values
gdf['cx'] = proj.geometry.centroid.x.values
gdf['cy'] = proj.geometry.centroid.y.values
gdf['is_condo'] = gdf['class'].isin(CONDO_CLASSES)
gdf['land_value'] = gdf['AssessedValueLand'].fillna(0.0)
gdf['improvement_value'] = gdf['AssessedValueBuildings'].fillna(0.0)

# Reference parcels: taxable rate-class, non-condo, with a land value and a real lot
ref = gdf[gdf['in_reform'] & ~gdf['is_condo'] & (gdf['land_value'] > 0) & (gdf['lot_sf'] > 300)]
ref_built = ref[ref['improvement_value'] > 0]
ref_land_psf = (ref['land_value'] / ref['lot_sf']).values
ref_land_share = (ref_built['land_value'] / ref_built['total_assmt']).values

# Floor: 10th-percentile land share of improved 6+ unit apartment and mixed-use buildings
dense = ref_built[ref_built['class'].isin(['03-610', '03-11+', '04-610', '04-11+'])]
LAND_SHARE_FLOOR = float((dense['land_value'] / dense['total_assmt']).quantile(0.10))

# Plat-lots holding a taxable building account with $0 land (every condo unit, plus a few building-only accounts)
gdf['plat_lot'] = gdf['PROPID'].str[:8]
landless = gdf['in_reform'] & (gdf['land_value'] <= 0) & (gdf['improvement_value'] > 0)
on_lot = gdf[gdf['plat_lot'].isin(gdf.loc[landless, 'plat_lot'])]   # every account on those lots, incl. exempt
footprints = gpd.GeoSeries(
    proj.loc[on_lot.index].geometry.groupby(on_lot['plat_lot']).apply(lambda s: s.union_all()), crs=3438)

lots = pd.DataFrame({
    'accounts': on_lot.groupby('plat_lot').size(),
    'landless_accounts': gdf.loc[landless].groupby('plat_lot').size(),
    'condo_units': on_lot[on_lot['is_condo']].groupby('plat_lot').size(),
    'value': on_lot.groupby('plat_lot')['total_assmt'].sum(),
    'booked_land': on_lot.groupby('plat_lot')['land_value'].sum(),
    'landless_value': gdf.loc[landless].groupby('plat_lot')['total_assmt'].sum(),
    'lot_sf': footprints.area,
    'cx': footprints.centroid.x,
    'cy': footprints.centroid.y,
}).fillna({'landless_accounts': 0, 'condo_units': 0})
lots['nb_land_psf'] = knn_median(lots['cx'].values, lots['cy'].values,
                                 ref['cx'].values, ref['cy'].values, ref_land_psf)
lots['nb_land_share'] = knn_median(lots['cx'].values, lots['cy'].values,
                                   ref_built['cx'].values, ref_built['cy'].values, ref_land_share)
ceiling = np.maximum(lots['nb_land_share'], LAND_SHARE_FLOOR)
lot_based = (lots['lot_sf'] * lots['nb_land_psf'] / lots['value'].replace(0, np.nan)).fillna(0.0)
lots['target_share'] = np.minimum(np.maximum(lot_based, LAND_SHARE_FLOOR), ceiling)
lots['bound'] = np.select([lot_based < LAND_SHARE_FLOOR, lot_based > ceiling], ['floor', 'ceiling'], default='lot-based')
# Under-landed: the land booked anywhere on the lot is below the plausible minimum share of the lot's value.
# Otherwise the land-less accounts are buildings on separately-assessed land (leased buildings): left as assessed.
lots['under_landed'] = lots['booked_land'] < LAND_SHARE_FLOOR * lots['value']
lots['missing_land'] = np.where(
    lots['under_landed'], (lots['target_share'] * lots['value'] - lots['booked_land']).clip(lower=0), 0.0)

# Allocate each lot's missing land to its land-less taxable accounts pro rata by value, capped at the lot ceiling
fix = landless & gdf['plat_lot'].map(lots['under_landed']).fillna(False).astype(bool)
alloc_share = gdf.loc[fix, 'plat_lot'].map(lots['missing_land'] / lots['landless_value'].replace(0, np.nan)).fillna(0.0)
alloc_share = np.minimum(alloc_share, gdf.loc[fix, 'plat_lot'].map(pd.Series(ceiling, index=lots.index)))
gdf['land_imputed'] = fix
gdf.loc[fix, 'land_value'] = (gdf.loc[fix, 'total_assmt'] * alloc_share).round(0)
gdf.loc[fix, 'improvement_value'] = gdf.loc[fix, 'total_assmt'] - gdf.loc[fix, 'land_value']

condo_units = gdf['in_reform'] & gdf['is_condo']
print(f"Land-share floor (p10 of improved 6+ unit buildings): {LAND_SHARE_FLOOR:.3f}")
print(f"Taxable condo units: {condo_units.sum():,}; with $0 land as assessed: "
      f"{(gdf.loc[condo_units, 'AssessedValueLand'].fillna(0) <= 0).mean():.1%}")
print(f"Plat-lots with land-less taxable buildings: {len(lots):,}; under-landed (imputed): {lots['under_landed'].sum():,}")
print(f"Accounts re-split: {fix.sum():,} ({(fix & gdf['is_condo']).sum():,} condo units, "
      f"{(fix & ~gdf['is_condo']).sum():,} other building accounts)")
print(lots.loc[lots['under_landed'], 'bound'].value_counts().to_string())
imputed = gdf.loc[fix]
print(f"Imputed land share: value-weighted {imputed['land_value'].sum() / imputed['total_assmt'].sum():.3f}, "
      f"median account {(imputed['land_value'] / imputed['total_assmt']).median():.3f}")
print(lots.sort_values('value', ascending=False).head(10)[
    ['accounts', 'condo_units', 'lot_sf', 'value', 'booked_land', 'nb_land_psf', 'target_share', 'bound',
     'under_landed']].round(3).to_string())
left = gdf[landless & ~fix]
print(f"\nLand-less building accounts left as assessed (land booked on another account of the lot): "
      f"{len(left):,}, ${left['total_assmt'].sum():,.0f} assessed")
print(left[['PROPID', 'short_desc', 'levy_code_1', 'total_assmt']].to_string(index=False))

In [ ]:
def providence_category(df: pd.DataFrame) -> pd.Series:
    # Roll class (authoritative for the tax year) -> standard LVTShift categories
    cls = df['class']
    units = df['NumUnits'].fillna(0)
    return pd.Series(np.select(
        [
            cls == '1',
            (cls == '2') & (units >= 5),
            cls == '2',
            cls.isin(['03-610', '03-11+']),
            cls.isin(['04-05U', '04-610', '04-11+']),
            cls == '23',
            cls == '24',
            cls.isin(['13', '14']),
            cls.isin(['5', '6']),
            cls == '7',
            cls == '10',
            cls == '33',
        ],
        [
            'Single Family Residential',
            'Large Multi-Family (5+ units)',
            'Small Multi-Family (2-4 units)',
            'Large Multi-Family (5+ units)',
            'Mixed Use',
            'Condominium',
            'Office / Commercial Condo',
            'Vacant Land',
            'Commercial',
            'Industrial',
            'Other Commercial',
            'Agricultural',
        ],
        default='Other',
    ), index=df.index)


gdf['PROPERTY_CATEGORY'] = providence_category(gdf)
# Override: $0 improvement -> Vacant Land
gdf.loc[gdf['improvement_value'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'
print(pd.crosstab(gdf.loc[gdf['in_reform'], 'PROPERTY_CATEGORY'], gdf.loc[gdf['in_reform'], 'levy_code_1']).to_string())

## Section 4 — Current Tax + Validation (Gate 1)

`current_tax` is the **actual FY2026 bill** (`total_taxes`). Rebuilding `(assessed − exemptions) × class rate` reproduces the
bills in every uniform-rate class. It overshoots only in `C01`, where Class 3 mixed-use ("combo") parcels are billed at a blend:
the residential rate on the residential portion and 29.20 on the rest (§ 21-182(c)(6)). Using the bills keeps each combo's blend.

Partial exemptions (veterans, 65+, disabled; § 21-126 value exemptions with capped dollar savings) are already reflected in the
bills. For the land/building split they are applied to improvements first, then land (LVTShift convention).

**Official control.** The audited FY2025 ACFR reports the real-property levy ($315,771K, Exhibit C-1). The city's FY2025 roll
(the 2024 Property Tax Roll), deduplicated and excluding TSA contract taxes, reproduces that figure; the city books TSA payments
separately as "PILOT & Tax Treaty (excluded from levy)". The same computation on the FY2026 roll is therefore the
official-equivalent FY2026 real-property levy. The modeled parcel base is checked against it; the gap is the re-platted accounts
dropped at the join.

In [ ]:
gdf['current_tax'] = gdf['total_taxes']
gdf['class_rate'] = gdf['levy_code_1'].map(RATE_CLASSES)
gdf['rebuilt_tax'] = gdf['net_assessed'] * gdf['class_rate'] / 1000

# Taxable values: partial exemptions applied to improvements first, then land
exempt = gdf['total_exempt'].clip(lower=0)
gdf['taxable_improvement_value'] = (gdf['improvement_value'] - exempt).clip(lower=0)
gdf['taxable_land_value'] = (gdf['land_value'] - (exempt - gdf['improvement_value']).clip(lower=0)).clip(lower=0)

rebuild = gdf[gdf['in_reform']].groupby('levy_code_1').agg(
    accounts=('PROPID', 'size'),
    rate=('class_rate', 'first'),
    net_assessed=('net_assessed', 'sum'),
    billed=('current_tax', 'sum'),
    rebuilt=('rebuilt_tax', 'sum'),
).reindex(list(RATE_CLASSES))
rebuild['rebuild_gap_pct'] = (rebuild['rebuilt'] / rebuild['billed'] - 1) * 100
print(rebuild.to_string(float_format=lambda v: f"{v:,.2f}"))

# Back-test: FY2025 roll (ex-TSA) vs the audited FY2025 ACFR real-property levy
prior_levy = roll_prior.loc[roll_prior['levy_code_1'] != 'TSA', 'total_taxes'].sum()
print(f"\nFY2025 back-test: roll ex-TSA ${prior_levy:,.0f} vs ACFR ${ACFR_FY2025_REAL_PROPERTY_LEVY:,.0f} "
      f"({(prior_levy / ACFR_FY2025_REAL_PROPERTY_LEVY - 1) * 100:+.3f}%)")

OFFICIAL_REVENUE = roll.loc[roll['levy_code_1'] != 'TSA', 'total_taxes'].sum()
modeled_levy = gdf.loc[gdf['levy_code_1'] != 'TSA', 'current_tax'].sum()
current_revenue = gdf.loc[gdf['in_reform'], 'current_tax'].sum()
gap_pct = (modeled_levy / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled: ${modeled_levy:,.0f}   Official: ${OFFICIAL_REVENUE:,.0f}   Gap: {gap_pct:+.2f}%")
print("  (official = FY2026 roll billed ex-TSA; modeled = the same on accounts that join to a GIS parcel)")
print(f"Reform base (7 rate classes, taxable): ${current_revenue:,.0f} "
      f"= {current_revenue / OFFICIAL_REVENUE:.1%} of the levy (the rest is 8LAW/PRA/R01, held constant)")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"

## Section 5 — Split-Rate Model (per rate class)

For each rate class *k* with ordinance rate *r_k*, a revenue-neutral 4:1 split is solved on that class's parcels alone, so every
class raises exactly what it raises today. Each parcel enters the solve at **rate-normalized** values: taxable land and building
× (its effective rate ÷ *r_k*). For uniform-rate parcels that factor is 1. For Class 3 mixed-use combos it is their billed blend
(combo 5U median ≈ 0.69), so a combo's residential portion keeps its residential discount under the reform. In effect each
parcel's new tax is `current_tax × (land share × L_k + building share × B_k) ÷ r_k`, where `L_k = 4 × B_k`.

Fully-exempt and out-of-scope accounts are held out of the solver and excluded from the export and charts (counted in Section 3).
The export needs a single `land_millage` / `improvement_millage`, so the per-class rates are averaged weighted by class revenue
(Rochester precedent). The per-class rates printed below are the actual result.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
n_out_of_scope = int((~gdf['in_reform'] & (gdf['full_exmp'] == 0)).sum())
gdf = gdf[gdf['in_reform']].copy()

gdf['rate_factor'] = (gdf['current_tax'] * 1000 / gdf['net_assessed']) / gdf['class_rate']
gdf['norm_land_value'] = gdf['taxable_land_value'] * gdf['rate_factor']
gdf['norm_improvement_value'] = gdf['taxable_improvement_value'] * gdf['rate_factor']
actual_taxable = gdf[['taxable_land_value', 'taxable_improvement_value']].copy()

class_results, parts = [], []
for levy_code, rate in RATE_CLASSES.items():
    part = gdf[gdf['levy_code_1'] == levy_code]
    land_m, imp_m, revenue, modeled = model_split_rate_tax(
        df=part,
        land_value_col='norm_land_value',
        improvement_value_col='norm_improvement_value',
        current_revenue=part['current_tax'].sum(),
        land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    )
    parts.append(modeled)
    class_results.append({
        'levy_code': levy_code, 'accounts': len(part), 'current_rate': rate,
        'land_millage': land_m, 'improvement_millage': imp_m,
        'current_revenue': part['current_tax'].sum(), 'new_revenue': revenue,
    })

gdf = pd.concat(parts).loc[gdf.index]
# model_split_rate_tax writes its (rate-normalized) inputs to taxable_*; restore the real taxable values
gdf[['taxable_land_value', 'taxable_improvement_value']] = actual_taxable

class_table = pd.DataFrame(class_results).set_index('levy_code')
weights = class_table['current_revenue'] / class_table['current_revenue'].sum()
land_millage = float((class_table['land_millage'] * weights).sum())
improvement_millage = float((class_table['improvement_millage'] * weights).sum())
new_revenue = gdf['new_tax'].sum()

print(f"Held out {n_exempt:,} fully-exempt + {n_out_of_scope:,} out-of-scope (TSA/8LAW/PRA/R01) accounts "
      f"(excluded from model, export, and charts).")
print(f"Modeled parcels: {len(gdf):,}")
print(class_table.to_string(float_format=lambda v: f"{v:,.2f}"))
print(f"Revenue-weighted land millage: {land_millage:.4f}   improvement millage: {improvement_millage:.4f}")
print(f"Revenue neutrality: ${new_revenue:,.0f} vs target ${current_revenue:,.0f} "
      f"(gap {(new_revenue / current_revenue - 1) * 100:+.4f}%)")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Providence — 4:1 Split-Rate Tax Impact (within each rate class)')

## Section 5b — Read the Results (Gate 5 Artifact Scan)

Check building shares by category against priors, read the results within each rate class (the reform is solved per class), and
confirm the condo fix: without it, condos would sit at their class's building-only floor.

In [ ]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
summary = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_low_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(summary.round(3).to_string())

print("\nMedian tax change % by category within each rate class:")
print(d.pivot_table(index='PROPERTY_CATEGORY', columns='levy_code_1', values='tax_change_pct',
                    aggfunc='median').reindex(columns=list(RATE_CLASSES)).round(1).to_string())

# Flat-ratio check (Maricopa canary): does single-family land share vary per parcel?
sfr = d[(d['PROPERTY_CATEGORY'] == 'Single Family Residential') & (d['improvement_value'] > 0)].copy()
sfr['land_share'] = sfr['land_value'] / (sfr['land_value'] + sfr['improvement_value'])
print(f"\nSFR land-share spread (flat-ratio check): p10={sfr['land_share'].quantile(.1):.3f}  "
      f"median={sfr['land_share'].median():.3f}  p90={sfr['land_share'].quantile(.9):.3f}  "
      f"distinct land values={sfr['land_value'].nunique():,}")

# Condo canary: with $0 land every condo would sit at its class's building-only floor
for levy_code in ['OO01', 'NO01', 'C01']:
    row = class_table.loc[levy_code]
    floor_pct = (row['improvement_millage'] / row['current_rate'] - 1) * 100
    cat = 'Office / Commercial Condo' if levy_code == 'C01' else 'Condominium'
    med = d.loc[(d['levy_code_1'] == levy_code) & (d['PROPERTY_CATEGORY'] == cat), 'tax_change_pct'].median()
    print(f"{levy_code} {cat}: building-only floor {floor_pct:+.1f}%  vs  imputed-land median {med:+.1f}%")

# Non-condo accounts re-split on under-landed lots, and leased buildings left building-only (at the floor by design)
print("\nNon-condo building accounts re-split on under-landed lots:")
print(d.loc[d['land_imputed'] & ~d['is_condo'],
            ['PROPID', 'short_desc', 'levy_code_1', 'total_assmt', 'land_value', 'tax_change_pct']].round(1).to_string(index=False))
leased = d[(d['land_value'] <= 0) & (d['improvement_value'] > 0)]
print(f"Leased buildings left building-only: {len(leased):,} accounts, ${leased['current_tax'].sum():,.0f} current tax, "
      f"median change {leased['tax_change_pct'].median():+.1f}%")

## Section 6 — Exploration Chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values().plot.barh(ax=ax)
ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'Providence — Median Tax Change % by Category ({LAND_IMPROVEMENT_RATIO:.0f}:1 split within each rate class)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('saved category_preview.png')

## Section 7 — Census Join + Export

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf already carries demographics — spatial join adds them above.
            # Do NOT do a second gdf.merge(census_data) here: census_gdf has the columns
            # baked in, so a second merge creates median_income_x/y duplicates and silently
            # zeros out all demographic output.
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet (geometry + tax outcomes + parcel id/owner/address) and a
# self-contained interactive HTML map colored by tax change. gdf must still carry geometry and
# the identity columns here — it does, since the model only ever row-filters gdf.
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)  # simplifies only the viewer geometry, not the parquet
print("Done.")

## Validation Summary

| Check | Result |
|---|---|
| Revenue match (Gate 1) | **−1.02%**: modeled $334.3M vs official-equivalent FY2026 real-property levy $337.8M (roll billed ex-TSA). The method back-tests at **+0.007%** against the audited FY2025 ACFR real-property levy ($315,771K). The gap is the 326 roll accounts whose lots were re-platted after the assessment date. Rate rebuild matches bills to ≤0.01% in 6 of 7 classes; C01 +3.95% is the Class 3 mixed-use blend, handled by using actual bills |
| Distribution (Gate 2) | Vacant Land +69.7% median; single-family −2.2%; 2–4 family −4.8%; 5+ units −9.3%; mixed use −13.7%; condos −27.6%. Revenue-neutral within every class to the dollar |
| Parcel count | 40,536 modeled accounts. Held out: 2,425 fully exempt, 133 TSA ($14.5M), 638 8LAW ($12.4M), 4 PRA/R01 ($8K). Dropped: 326 unjoined roll accounts ($3.8M billed) and 201 blank-PROPID GIS slivers ($0) |
| Census coverage (Gate 3) | 100% of parcels matched to a block group; median income present for 83.8% (ACS suppresses income in some block groups, e.g. around the colleges) |
| PNGs generated (Gate 4) | 7 of 7 in `analysis/reports/providence/` |
| Parcel map | `analysis/maps/providence.parquet` (40,536 parcels; `parcel_id`, owner and situs address 100% populated) + self-contained `parcel_map.html`. `parcel_url` N/A: deliberate `None`, no verified Vision deep link |
| Artifact scan (Gate 5) | **Condo $0-land artifact found and fixed**: 4,290 condo units + 9 other building accounts on 673 under-landed plat-lots re-split (value-weighted imputed land share 23%); without it condos sat at the building-only floor (−57%). 45 leased-land building accounts ($0.9M tax) are correctly building-only because their land is assessed on a separate ground account. SFR land share varies per parcel (p10 0.33 / median 0.42 / p90 0.59, 2,600 distinct land values): no flat-ratio artifact. No ceiling clustering: the only other large increase is Other Commercial (34 utility parcels, 8% median building share), a real land-heavy result |
| SFR median change | −2.2% |
| Vacant land median change | +69.7% |

**Per-class solved rates (per $1,000):**

| Class | Current | Land | Building |
|---|---|---|---|
| OO01 owner-occ. 1 unit | 8.40 | 14.37 | 3.59 |
| OO2-5 owner-occ. 2–5 | 7.55 | 15.47 | 3.87 |
| NO01 non-owner-occ. 1 unit | 14.60 | 24.78 | 6.19 |
| NOO2-5 non-owner-occ. 2–5 | 14.00 | 28.71 | 7.18 |
| C610 6–10 units | 26.00 | 68.72 | 17.18 |
| C11 11+ units | 28.50 | 78.78 | 19.69 |
| C01 commercial/industrial | 29.20 | 52.29 | 13.07 |

The export's single `land_millage` / `improvement_millage` (36.49 / 9.12) is the revenue-weighted average of these.

### Modeling choices & limitations

- **Condo results depend on the land imputation.** The assessor books no land for condos, so their land share is modeled
  (Section 3). At the imputed ~24% share, a condo's median change is about −27%. At the 2–5 family median (28%) it would be about
  −21%; under the rejected neighbor-share method (~45%) about 0%; at the floor (8%) about −47%. The chosen method follows lot
  density. Condos fall more than 2–5 family buildings mainly because Class 1A pairs building-heavy condos with land-heavy
  single-family homes and residential vacant lots, and the split is solved within the class.
- **Within-class solve.** No revenue moves between rate classes, so the owner-occupied discount and the commercial premium are
  preserved exactly. A citywide split would produce different (larger) shifts.
- **Class 3 mixed-use** combos stay in C01's solve with rate-normalized values. Their residential and commercial portions are not
  separated, because the data does not say how much of each combo is residential.
- **Held constant:** TSA contract taxes, 8LAW rent-based taxes, and tangible personal property (Class 4) are outside the reform.
- **Commercial detail:** the GIS carries no finer commercial use code, so retail, office, and hotel are not separated.
- **Legality** is not assessed here. This is an illustrative revenue-neutral restructuring; see `/legality-analyzer` for Rhode
  Island's authority to tax land and buildings at different rates.